In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = pd.read_csv("../data/healthcare-dataset-stroke-data.csv")
df = df.drop(columns="id")
df = df[df["gender"] != "Other"].reset_index(drop=True)
df["bmi_missing"] = df["bmi"].isna().astype(int)

X = df.drop(columns="stroke")
y = df["stroke"]

train/test split

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)
print("Stroke rate train:", y_train.mean().round(3), "| test:", y_test.mean().round(3))
print("Strokes train:", y_train.sum(), "| test:", y_test.sum())

(4087, 11) (1022, 11)
Stroke rate train: 0.049 | test: 0.049
Strokes train: 199 | test: 50


In [3]:
num_cols = ["age", "avg_glucose_level", "bmi"]
bin_cols = ["hypertension", "heart_disease", "bmi_missing"]
cat_cols = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
cat_pipe = OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False)

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("bin", "passthrough", bin_cols),
    ("cat", cat_pipe, cat_cols),
])

In [4]:
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print(X_train_prep.shape, X_test_prep.shape)
print(preprocessor.get_feature_names_out())
print("NaN left:", np.isnan(X_train_prep).sum(), np.isnan(X_test_prep).sum())
print("Median learned:", preprocessor.named_transformers_["num"].named_steps["imputer"].statistics_)
print("Train BMI median:", X_train["bmi"].median())

(4087, 18) (1022, 18)
['num__age' 'num__avg_glucose_level' 'num__bmi' 'bin__hypertension'
 'bin__heart_disease' 'bin__bmi_missing' 'cat__gender_Male'
 'cat__ever_married_Yes' 'cat__work_type_Govt_job'
 'cat__work_type_Never_worked' 'cat__work_type_Private'
 'cat__work_type_Self-employed' 'cat__work_type_children'
 'cat__Residence_type_Urban' 'cat__smoking_status_Unknown'
 'cat__smoking_status_formerly smoked' 'cat__smoking_status_never smoked'
 'cat__smoking_status_smokes']
NaN left: 0 0
Median learned: [45.   91.89 28.1 ]
Train BMI median: 28.1
